## Imports

In [1]:
import os
import re
import mne
import pathlib
import shutil
from tqdm import tqdm
import pandas as pd
from pathlib import Path

## Set current dir to project root dir

In [2]:
def find_project_root():
    """Walk up from CWD until we find the project root."""
    markers = [".git", "Makefile", "renv.lock", ".Rprofile"]
    path = Path.cwd()
    while path != path.parent:
        if any((path / m).exists() for m in markers):
            return path
        path = path.parent
    raise FileNotFoundError("Could not find project root")

os.chdir(find_project_root())

## Functions

In [3]:
def get_subject_folders(parent_directory):
    """
    Returns a list of paths for all subfolders 
    starting with 'sub-' in the given directory.
    """
    path = Path(parent_directory)
    
    # .glob('sub-*') looks for items starting with 'sub-'
    # is_dir() ensures we only get folders, not files
    subject_folders = [str(f) for f in path.glob('sub-*') if f.is_dir()]
    
    return sorted(subject_folders)

# Example Usage:
# folders = get_subject_folders('/path/to/your/eeg_data')
# print(f"Found {len(folders)} subject folders.")

In [4]:
def extract_unique_stimuli(file_path):
    """
    Parses a BrainVision .vmrk file and returns a sorted list 
    of all unique stimulus descriptions (trigger codes).
    """
    stimuli = set()
    
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            for line in file:
                # Look for lines starting with Mk followed by digits (e.g., Mk100=...)
                if line.startswith('Mk'):
                    # Split by '=' to separate key and values, then split values by ','
                    try:
                        # Format: Mk<Num>=Type,Description,Position,Size,Channel
                        content = line.split('=')[1]
                        parts = content.split(',')
                        
                        marker_type = parts[0].strip()
                        description = parts[1].strip()
                        
                        # Only add if the type is exactly 'Stimulus'
                        if marker_type == 'Stimulus':
                            stimuli.add(description)
                    except (IndexError, ValueError):
                        # Skip malformed lines
                        continue
                        
    except FileNotFoundError:
        return "Error: File not found."

    # Return as a sorted list for easier viewing
    return sorted(list(stimuli))

## Variables

## Main

In [5]:
main_data_folder = "./ds006018"
tasks = ["task-auditoryoddball", "task-flanker", "task-visualoddball", "task-visualsearch"]

In [6]:
paths_to_original_data_actors = get_subject_folders(main_data_folder)

In [7]:
for i in tqdm(range(len(paths_to_original_data_actors))):

    for task_no in range(len(tasks)):

        task = tasks[task_no]

        sub_number = paths_to_original_data_actors[i].split('/')[-1]

        path_to_vhdr = paths_to_original_data_actors[i] + "/eeg/" + sub_number + "_" + task + "_eeg.vhdr"
        path_to_vmrk = paths_to_original_data_actors[i] + "/eeg/" + sub_number + "_" + task + "_eeg.vmrk"

        directory = Path("./ds006018_per_stimuli/"+sub_number)
        directory.mkdir(parents=True, exist_ok=True)

        file_path = Path(path_to_vhdr)

        if file_path.is_file():
            print("The file exists!")
        


            # Get unique stimulus numbers from the .vmrk file
            unique_stimuli_numers = extract_unique_stimuli(path_to_vmrk)
            print(unique_stimuli_numers)


            # 1. Load the BrainVision data
            raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)


            # 2. Set the montage (Standard 10-20 system for electrode locations)
            montage = mne.channels.make_standard_montage('standard_1020')
            raw.set_montage(montage)


            # 3. Filtering (Standard for EEG: 0.1Hz to 40Hz)
            raw.filter(l_freq=0.1, h_freq=40.0)


            # 4. Plotting the data to inspect for noise
            #raw.plot(n_channels=15, duration=5, scalings='auto')


            # 5. Preprocessing (Required for FDA to reduce noise)
            #raw.resample(200)               # Downsample to reduce R processing time - let's do this since we have a lot of data and FDA can be computationally intensive

            # 6. Create Epochs (FDA usually analyzes trials/segments) - by 
            # This assumes you have event markers in your .vmrk file
            events, event_id = mne.events_from_annotations(raw)

            stimulus_dataframes = {}

            # Iterate through every stimulus
            for event_name, event_val in event_id.items():
                # Clean the name for filenames (e.g., 'Stimulus_S1')
                clean_name = event_name.replace('/', '_').replace(' ', '')
                
                try:
                    # Pass a dictionary where the key is the name and value is the integer ID
                    # This resolves the "must be an int, got str" error
                    current_epochs = mne.Epochs(raw, events, event_id={event_name: event_val},
                                                tmin=-0.2, tmax=0.8, preload=True)
                    
                    if len(current_epochs) > 0:
                        df_temp = current_epochs.to_data_frame()
                        # Drop 'condition' column and standardise column order to match R/eegUtils output
                        df_temp = df_temp.drop(columns=["condition"], errors="ignore")
                        df_temp = df_temp[["time", "epoch"] + [c for c in df_temp.columns if c not in ["time", "epoch"]]]
                        
                        # Store and export
                        stimulus_dataframes[clean_name] = df_temp
                        df_temp.to_csv("./ds006018_per_stimuli/"+sub_number+"/"+task+"_"+clean_name+".csv", index=False)
                        
                        print(f"Successfully created: eeg_{clean_name}.csv ({len(current_epochs)} trials)")
                        
                        
                except Exception as e:
                    print(f"Skipping {event_name}: {e}")

        else:
            print(path_to_vhdr+"File not found.")

  0%|          | 0/127 [00:00<?, ?it/s]

The file exists!
['S  1', 'S 70', 'S 80', 'S180']
Extracting parameters from ds006018/sub-001/eeg/sub-001_task-auditoryoddball_eeg.vhdr...
Setting channel info structure...
Reading 0 ... 124029  =      0.000 ...   248.058 secs...


/tmp/ipykernel_832806/1130332472.py:28: RuntimeWarning: No coordinate information found for channels ['HEL', 'LM', 'RM', 'HER', 'VER']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)
/tmp/ipykernel_832806/1130332472.py:28: RuntimeWarning: Not setting positions of 5 misc channels found in montage:
['HEL', 'LM', 'RM', 'HER', 'VER']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 16501 samples (33.002 s)

Used Annotations descriptions: [np.str_('Stimulus/S  1'), np.str_('Stimulus/S 70'), np.str_('Stimulus/S 80'), np.str_('Stimulus/S180')]
Not setting metadata
1 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 1 events and 501 original time points ...
0 bad epochs dropped
Successfully created: eeg_Stimulus_S1.csv 

/tmp/ipykernel_832806/1130332472.py:28: RuntimeWarning: No coordinate information found for channels ['HEL', 'LM', 'RM', 'HER', 'VER']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)
/tmp/ipykernel_832806/1130332472.py:28: RuntimeWarning: Not setting positions of 5 misc channels found in montage:
['HEL', 'LM', 'RM', 'HER', 'VER']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)


Used Annotations descriptions: [np.str_('Stimulus/S  2'), np.str_('Stimulus/S 11'), np.str_('Stimulus/S 12'), np.str_('Stimulus/S 21'), np.str_('Stimulus/S 22'), np.str_('Stimulus/S111'), np.str_('Stimulus/S112'), np.str_('Stimulus/S121'), np.str_('Stimulus/S122'), np.str_('Stimulus/S211'), np.str_('Stimulus/S212'), np.str_('Stimulus/S221'), np.str_('Stimulus/S222')]
Not setting metadata
1 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 1 events and 501 original time points ...
0 bad epochs dropped
Successfully created: eeg_Stimulus_S2.csv (1 trials)
Not setting metadata
100 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 100 events and 501 original time points ...
0 bad epochs dropped
Successfully created: eeg_Stimulus_S11.csv (100 trials)
Not s

/tmp/ipykernel_832806/1130332472.py:28: RuntimeWarning: No coordinate information found for channels ['HEL', 'LM', 'RM', 'HER', 'VER']. Setting channel types to misc. To avoid this warning, set channel types explicitly.
  raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)
/tmp/ipykernel_832806/1130332472.py:28: RuntimeWarning: Not setting positions of 5 misc channels found in montage:
['HEL', 'LM', 'RM', 'HER', 'VER']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_brainvision(path_to_vhdr, preload=True)


Used Annotations descriptions: [np.str_('Stimulus/S 11'), np.str_('Stimulus/S 12'), np.str_('Stimulus/S 13'), np.str_('Stimulus/S 14'), np.str_('Stimulus/S 15'), np.str_('Stimulus/S 21'), np.str_('Stimulus/S 22'), np.str_('Stimulus/S 23'), np.str_('Stimulus/S 24'), np.str_('Stimulus/S 25'), np.str_('Stimulus/S 31'), np.str_('Stimulus/S 32'), np.str_('Stimulus/S 33'), np.str_('Stimulus/S 34'), np.str_('Stimulus/S 35'), np.str_('Stimulus/S 41'), np.str_('Stimulus/S 42'), np.str_('Stimulus/S 43'), np.str_('Stimulus/S 44'), np.str_('Stimulus/S 45'), np.str_('Stimulus/S 51'), np.str_('Stimulus/S 52'), np.str_('Stimulus/S 53'), np.str_('Stimulus/S 54'), np.str_('Stimulus/S 55'), np.str_('Stimulus/S201'), np.str_('Stimulus/S202')]
Not setting metadata
8 matching events found
Setting baseline interval to [-0.2, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 8 events and 501 original time points ...
0 bad epochs dropped
Successful

  0%|          | 0/127 [00:29<?, ?it/s]


KeyboardInterrupt: 